# Cow lameness detection with a small temporal CNN

Keypoint sequences (20 keypoints per frame, 10 fps) -> per-video lame / normal score.

Expected folder layout (one CSV per video):
```
HOME/
  Lame/    *.csv    -> label 1 (anomaly)
  Normal/  *.csv    -> label 0
```
Each CSV has columns `image, keypoint, x, y, score`. The extra `x_full` / `y_full` columns from the rescaling step are dropped on load.

Set `HOME` in the config cell, then run top to bottom.

## Imports

In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.signal import savgol_filter
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset

## 0. Config
Everything you might want to change lives here.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
HOME = Path("/content/drive/MyDrive/hackathon/Data/data")   # folder that contains Lame/ and Normal/
OUT_DIR = Path("outputs")

# Folder name -> label. Lame is the positive (anomaly) class.
CLASSES = {"Normal": 0, "Lame": 1}

# CSV format
HAS_HEADER = True          # set False if the CSVs have no header row
DROP_COLS = ["x_full", "y_full"]   # full-frame coords added by the rescaling step; not used here

# Keypoints (indices follow your pose-visualization image)
K = 20                     # keypoints per frame
TAIL, WITHERS, NOSE = 6, 7, 4   # 6 = tail base, 7 = top of back, 4 = nose
CONF_THR = 0.3             # keypoints below this score are treated as missing
SMOOTH_WIN, SMOOTH_ORDER = 5, 2  # Savitzky-Golay; short because gait cycle is only ~10-15 frames at 10 fps

# Windowing
W = 25                     # window length in frames (3 s at 10 fps)
TEST_STRIDE = 5            # step between windows when scoring a whole video
TOPK = None                # None = mean of window probabilities; int k = mean of the k highest

# Model / training
HIDDEN, KERNEL, DROPOUT = 32, 5, 0.3
EPOCHS = 60                # fixed; do not tune on the test folds
WINDOWS_PER_EPOCH = 400    # random crops drawn per epoch
BATCH = 32
LR, WD = 1e-3, 1e-2
JITTER = 0.01              # gaussian noise std, in body-length units

# Evaluation
N_SPLITS = 5               # cross-validation folds (reduced automatically if a class is small)
N_REPEATS = 3              # repeat CV with different fold assignments
SEED = 0
TRAIN_FINAL = True         # also train one model on all data and save it

In [4]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## 1. Loading one video
CSV -> array of shape `(T frames, K keypoints, 3)`. `x_full` / `y_full` are dropped here.

In [5]:
def load_video(path):
    df = pd.read_csv(path, sep=None, engine="python",       # sep=None auto-detects tab/comma
                     header=0 if HAS_HEADER else None)
    if HAS_HEADER:
        df.columns = df.columns.astype(str).str.strip()
        df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])   # drop x_full / y_full
    df = df.iloc[:, :5]
    df.columns = ["image", "keypoint", "x", "y", "score"]

    # Frame number = last run of digits in the image name ("000012.jpg" -> 12)
    frame = df["image"].astype(str).str.extract(r"(\d+)(?=\D*$)")[0].astype(int).values
    frame = frame - frame.min()

    # Frames/keypoints with no row stay NaN and get filled in preprocessing
    arr = np.full((frame.max() + 1, K, 3), np.nan)
    arr[frame, df["keypoint"].astype(int).values] = df[["x", "y", "score"]].values
    return arr

## 2. Preprocessing one video
`(T, K, 3)` -> feature matrix `(T, 102)`. Everything is done per video, so nothing spans two videos.

In [6]:
def preprocess(arr, name=""):
    xy = arr[..., :2].copy()
    score = np.nan_to_num(arr[..., 2])            # missing rows get score 0

    # 2a. Treat low-confidence keypoints as missing, then fill the gaps by
    #     interpolating each keypoint's x and y over time. No frames are added;
    #     this only fills values inside the video's own time axis.
    xy[score < CONF_THR] = np.nan
    for k in range(K):
        for c in range(2):
            xy[:, k, c] = pd.Series(xy[:, k, c]).interpolate(limit_direction="both").values
    if np.isnan(xy).any():                         # a keypoint was never confident
        print(f"  warning: {name}: a keypoint never exceeded the confidence threshold")
        xy = np.nan_to_num(xy)

    # 2b. Light temporal smoothing (window kept short so the gait is not blurred)
    if len(xy) >= SMOOTH_WIN:
        xy = savgol_filter(xy, SMOOTH_WIN, SMOOTH_ORDER, axis=0)

    # 2c. Make every cow face right, so videos are comparable
    if np.mean(xy[:, NOSE, 0] - xy[:, TAIL, 0]) < 0:
        xy[..., 0] *= -1

    # 2d. Remove camera position and distance:
    #     origin = midpoint of tail base and top of back (per frame),
    #     scale  = mean distance between those two points over the whole video.
    root = (xy[:, TAIL] + xy[:, WITHERS]) / 2
    body_len = np.linalg.norm(xy[:, WITHERS] - xy[:, TAIL], axis=-1).mean() + 1e-6
    xy = (xy - root[:, None]) / body_len

    # 2e. Velocities: joint motion relative to the body, and the body's own
    #     speed (in body lengths per frame), which the origin shift removed.
    vel = np.diff(xy, axis=0, prepend=xy[:1])
    root_vel = np.diff(root / body_len, axis=0, prepend=root[:1] / body_len)

    # 2f. Stack channels: 40 positions + 40 velocities + 20 scores + 2 body velocity = 102
    feats = np.concatenate([xy.reshape(len(xy), -1),
                            vel.reshape(len(xy), -1),
                            score,
                            root_vel], axis=1)
    return feats.astype(np.float32)

## 3. Building the dataset
Walk the Lame / Normal folders, label by folder.

In [7]:
def build_records(home):
    records = []
    for cls_name, label in CLASSES.items():
        cls_dir = home / cls_name
        if not cls_dir.is_dir():
            raise FileNotFoundError(f"Expected folder not found: {cls_dir}")
        for p in sorted(cls_dir.rglob("*.csv")):
            rel = str(p.relative_to(home).with_suffix(""))
            records.append({
                "name": rel,
                "feats": preprocess(load_video(p), name=p.name),
                "label": label,
                # Group id used to keep related videos in the same CV fold.
                # Default: each file is its own group. If several files come from
                # the same cow or the same source video, change this to a shared id
                # (e.g. p.parent.name or a cow id parsed from the filename).
                "group": rel,
            })
    n_pos = sum(r["label"] for r in records)
    print(f"{len(records)} videos: {n_pos} Lame, {len(records) - n_pos} Normal")
    return records

In [8]:
def drop_short(records):
    # Windows must fit inside a video; shorter videos are dropped (never padded).
    # If many are dropped, lower W instead.
    kept = [r for r in records if len(r["feats"]) >= W]
    for r in records:
        if len(r["feats"]) < W:
            print(f"  dropped (only {len(r['feats'])} frames < W={W}): {r['name']}")
    print(f"kept {len(kept)} of {len(records)} videos")
    labels = [r["label"] for r in kept]
    if min(sum(labels), len(labels) - sum(labels)) < 2:
        raise ValueError("Need at least 2 videos of each class after dropping short ones.")
    return kept

## 4. Training data
Random fixed-length windows cropped from random videos. Frames inside a window keep their order; windows never cross a video boundary.

In [9]:
class TrainWindows(Dataset):
    def __init__(self, records, n_samples=WINDOWS_PER_EPOCH):
        self.records, self.n = records, n_samples

    def __len__(self):
        return self.n

    def __getitem__(self, _):
        r = self.records[np.random.randint(len(self.records))]   # pick a video
        f = r["feats"]
        s = np.random.randint(0, len(f) - W + 1)                 # pick a start inside it
        noise = np.random.normal(0, JITTER, (W, f.shape[1])).astype(np.float32)
        x = f[s:s + W] + noise                                   # (W, channels) + jitter
        return torch.from_numpy(x.T.copy()), torch.tensor(float(r["label"]))  # (channels, W)

## 5. Model
Small temporal CNN. Input `(batch, channels, W)`; global mean+max pooling makes it work for any window length. Receptive field: 17 frames (~1.7 s).

In [10]:
class SmallTCN(nn.Module):
    def __init__(self, in_ch, hidden=HIDDEN, k=KERNEL, dropout=DROPOUT):
        super().__init__()
        p = k // 2
        self.net = nn.Sequential(
            nn.BatchNorm1d(in_ch),                                   # standardize input channels
            nn.Conv1d(in_ch, hidden, k, padding=p),
            nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(hidden, hidden, k, padding=p, stride=2),       # downsample in time
            nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(hidden, hidden, k, padding=p, stride=2),
            nn.BatchNorm1d(hidden), nn.ReLU(),
        )
        self.head = nn.Linear(2 * hidden, 1)

    def forward(self, x):                       # x: (batch, channels, time)
        h = self.net(x)                         # (batch, hidden, time')
        h = torch.cat([h.mean(-1), h.amax(-1)], dim=1)   # global average + max pooling
        return self.head(h).squeeze(1)          # one logit per window

## 6. Scoring a whole video
Slide windows over it, average the probabilities.

In [11]:
@torch.no_grad()
def predict_video(model, feats, device, stride=TEST_STRIDE):
    model.eval()
    starts = list(range(0, len(feats) - W + 1, stride))
    if starts[-1] != len(feats) - W:            # make sure the final frames are covered
        starts.append(len(feats) - W)
    x = torch.from_numpy(np.stack([feats[s:s + W].T for s in starts])).to(device)
    p = torch.sigmoid(model(x)).cpu().numpy()   # one probability per window
    if TOPK:                                    # useful if anomalies are brief
        return float(np.sort(p)[-TOPK:].mean())
    return float(p.mean())

## 7. Training one model

In [12]:
def train_model(train_records, device, seed):
    set_seed(seed)
    in_ch = train_records[0]["feats"].shape[1]
    model = SmallTCN(in_ch).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    # Class imbalance: weight the positive (Lame) class by neg/pos ratio
    n_pos = sum(r["label"] for r in train_records)
    n_neg = len(train_records) - n_pos
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(n_neg / max(n_pos, 1), device=device))

    loader = DataLoader(TrainWindows(train_records), batch_size=BATCH, num_workers=0)
    for _ in range(EPOCHS):                     # fixed number of epochs, no early stopping
        model.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss_fn(model(x), y).backward()
            opt.step()
    return model

## 8. Cross-validation
Stratified by label, grouped so related videos stay together. Every video is scored exactly once, by a model that never saw its group.

In [13]:
def cross_validate(records, seed, device):
    labels = np.array([r["label"] for r in records])
    groups = np.array([r["group"] for r in records])
    n_splits = int(min(N_SPLITS, labels.sum(), len(labels) - labels.sum()))
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof = np.full(len(records), np.nan)         # out-of-fold probability per video
    for fold, (tr, te) in enumerate(cv.split(np.zeros(len(labels)), labels, groups)):
        model = train_model([records[i] for i in tr], device, seed + fold)
        for i in te:
            oof[i] = predict_video(model, records[i]["feats"], device)
        print(f"  fold {fold + 1}/{n_splits}: trained on {len(tr)} videos, scored {len(te)}")
    return oof

## 9. Reporting
Metrics per video, plus a CSV of every video's score.

In [14]:
def report(records, oofs):
    labels = np.array([r["label"] for r in records])
    aucs = [roc_auc_score(labels, o) for o in oofs]
    print(f"\nvideo-level ROC AUC per repeat: {np.round(aucs, 3)}"
          f"  (mean {np.mean(aucs):.3f} +/- {np.std(aucs):.3f})")

    mean_oof = np.mean(oofs, axis=0)            # average over repeats
    pred = (mean_oof >= 0.5).astype(int)        # note: pos_weight shifts probabilities upward
    print(f"at threshold 0.5: accuracy {accuracy_score(labels, pred):.3f} | "
          f"precision {precision_score(labels, pred, zero_division=0):.3f} | "
          f"recall {recall_score(labels, pred, zero_division=0):.3f} | "
          f"F1 {f1_score(labels, pred, zero_division=0):.3f}")
    print("confusion matrix (rows = true Normal/Lame, cols = predicted Normal/Lame):")
    print(confusion_matrix(labels, pred, labels=[0, 1]))

    df = pd.DataFrame({"video": [r["name"] for r in records],
                       "group": [r["group"] for r in records],
                       "label": labels,
                       "prob_lame": mean_oof})
    df.sort_values("prob_lame", ascending=False).to_csv(OUT_DIR / "oof_predictions.csv", index=False)
    print(f"saved per-video scores to {OUT_DIR / 'oof_predictions.csv'}")

## 10. Run

### Load data

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"device: {device}")

records = drop_short(build_records(HOME))

device: cuda
59 videos: 22 Lame, 37 Normal
  dropped (only 19 frames < W=25): Normal/N (18)
  dropped (only 14 frames < W=25): Normal/N (23)
kept 57 of 59 videos


### Cross-validation

In [16]:
oofs = []
for repeat in range(N_REPEATS):
    print(f"\ncross-validation repeat {repeat + 1}/{N_REPEATS}")
    oofs.append(cross_validate(records, SEED + 100 * repeat, device))


cross-validation repeat 1/3
  fold 1/5: trained on 46 videos, scored 11
  fold 2/5: trained on 46 videos, scored 11
  fold 3/5: trained on 45 videos, scored 12
  fold 4/5: trained on 45 videos, scored 12
  fold 5/5: trained on 46 videos, scored 11

cross-validation repeat 2/3
  fold 1/5: trained on 46 videos, scored 11
  fold 2/5: trained on 46 videos, scored 11
  fold 3/5: trained on 46 videos, scored 11
  fold 4/5: trained on 45 videos, scored 12
  fold 5/5: trained on 45 videos, scored 12

cross-validation repeat 3/3
  fold 1/5: trained on 46 videos, scored 11
  fold 2/5: trained on 46 videos, scored 11
  fold 3/5: trained on 45 videos, scored 12
  fold 4/5: trained on 46 videos, scored 11
  fold 5/5: trained on 45 videos, scored 12


### Report

In [17]:
report(records, oofs)


video-level ROC AUC per repeat: [0.894 0.83  0.864]  (mean 0.862 +/- 0.026)
at threshold 0.5: accuracy 0.842 | precision 0.783 | recall 0.818 | F1 0.800
confusion matrix (rows = true Normal/Lame, cols = predicted Normal/Lame):
[[30  5]
 [ 4 18]]
saved per-video scores to outputs/oof_predictions.csv


### Final model
Trained on all videos; this is the one you would deploy.

In [18]:
if TRAIN_FINAL:
    model = train_model(records, device, SEED)
    torch.save({"state_dict": model.state_dict(),
                "in_ch": records[0]["feats"].shape[1],
                "window": W}, OUT_DIR / "final_model.pt")
    print(f"saved final model to {OUT_DIR / 'final_model.pt'}")

saved final model to outputs/final_model.pt
